In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，加载数据和CV设置
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

Path('models/dnn').mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)
with open('data/cv_setup_inner.pkl', 'rb') as f:
    cv_setup_inner = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

# 超参数候选列表
param_combinations = [
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.2, 'learning_rate': 0.01,  'batch_size': 32},
    {'hidden_layers': [64, 32, 16], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [32, 16],     'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32},
    {'hidden_layers': [64, 32],     'dropout_rate': 0.3, 'learning_rate': 0.001, 'batch_size': 32},
]

EPOCHS   = 200
PATIENCE = 20
VERBOSE  = 0

EDGE_THRESHOLD   = 0.1
ATOM_TYPE_COLS   = ['T_O', 'T_C', 'T_Ti1', 'T_Ti2']
ATOM_TYPE_LABELS = ['O', 'C', 'Ti_inner', 'Ti_outer']

# =============================================================================
# 辅助函数
# =============================================================================

def create_dnn_model(input_dim, hidden_layers, dropout_rate, learning_rate):
    model = Sequential()
    model.add(Dense(hidden_layers[0], activation='relu', input_dim=input_dim))
    model.add(Dropout(dropout_rate))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation='relu'))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse', metrics=['mae'])
    return model

def calculate_metrics(y_true, y_pred):
    return {
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2':        float(r2_score(y_true, y_pred)),
        'n_samples': int(len(y_true))
    }

def classify_and_evaluate(y_true, y_pred, df_subset):
    results = {}
    results['overall'] = calculate_metrics(y_true, y_pred)

    edge_mask     = df_subset['D_E'].values < EDGE_THRESHOLD
    interior_mask = ~edge_mask

    results['edge']     = calculate_metrics(y_true[edge_mask],     y_pred[edge_mask])     if edge_mask.sum()     > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    results['interior'] = calculate_metrics(y_true[interior_mask], y_pred[interior_mask]) if interior_mask.sum() > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    results['by_atom_type'] = {}
    for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
        mask = df_subset[col].values == 1
        results['by_atom_type'][label] = calculate_metrics(y_true[mask], y_pred[mask]) if mask.sum() > 0 \
            else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    return results

def apply_charge_neutrality_correction(df_predictions):
    """按电荷绝对值加权，对每个结构应用电中性校正"""
    df_corrected = df_predictions.copy()
    df_corrected['y_pred_corrected']  = 0.0
    df_corrected['correction_amount'] = 0.0
    df_corrected['correction_weight'] = 0.0
    df_corrected['Q_total_before']    = 0.0

    for structure_id in df_corrected['structure_id'].unique():
        mask    = df_corrected['structure_id'] == structure_id
        charges = df_corrected.loc[mask, 'y_pred'].values
        Q_total = charges.sum()

        abs_charges     = np.abs(charges)
        sum_abs_charges = abs_charges.sum()
        weights = abs_charges / sum_abs_charges if sum_abs_charges > 1e-10 \
                  else np.ones(len(charges)) / len(charges)

        corrections = weights * Q_total
        df_corrected.loc[mask, 'y_pred_corrected']  = charges - corrections
        df_corrected.loc[mask, 'correction_amount'] = corrections
        df_corrected.loc[mask, 'correction_weight'] = weights
        df_corrected.loc[mask, 'Q_total_before']    = Q_total

    df_corrected['error_corrected']     = df_corrected['y_pred_corrected'] - df_corrected['y_true']
    df_corrected['abs_error_corrected'] = np.abs(df_corrected['error_corrected'])
    return df_corrected




In [2]:
# =============================================================================
# 嵌套交叉验证主循环
# =============================================================================

outer_results          = []
best_models            = []
all_train_predictions  = []
all_test_predictions   = []
training_histories     = []   # DNN特有：保存各折训练历史
classification_results = {'fold_wise': {}, 'averaged': {}}

start_time = time.time()

for fold_idx, outer_fold in enumerate(outer_folds):
    train_idx = outer_fold['train_atoms_idx']
    test_idx  = outer_fold['test_atoms_idx']

    X_train = X[train_idx];  y_train = y[train_idx]
    X_test  = X[test_idx];   y_test  = y[test_idx]

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    inner_folds = cv_setup_inner[fold_idx]['inner_folds']

    # 内层CV：遍历超参数候选，选最小平均验证MSE
    best_params      = None
    best_inner_score = float('inf')

    for params in param_combinations:
        inner_scores = []
        for inner_fold in inner_folds:
            inner_train_set = set(inner_fold['inner_train_structures'])
            inner_val_set   = set(inner_fold['inner_val_structures'])
            struct_ids = df.iloc[train_idx]['structure_id'].values

            inner_train_idx = np.where([s in inner_train_set for s in struct_ids])[0]
            inner_val_idx   = np.where([s in inner_val_set   for s in struct_ids])[0]

            model = create_dnn_model(X.shape[1], **{k: params[k] for k in
                                                    ['hidden_layers','dropout_rate','learning_rate']})
            model.fit(X_train_scaled[inner_train_idx], y_train[inner_train_idx],
                      validation_data=(X_train_scaled[inner_val_idx], y_train[inner_val_idx]),
                      epochs=EPOCHS, batch_size=params['batch_size'],
                      callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                                         restore_best_weights=True, verbose=0)],
                      verbose=VERBOSE)
            inner_scores.append(model.evaluate(X_train_scaled[inner_val_idx],
                                               y_train[inner_val_idx], verbose=0)[0])

        mean_score = np.mean(inner_scores)
        if mean_score < best_inner_score:
            best_inner_score = mean_score
            best_params      = params

    # 用最优参数在外层训练集训练
    best_model = create_dnn_model(X.shape[1], **{k: best_params[k] for k in
                                                  ['hidden_layers','dropout_rate','learning_rate']})
    history = best_model.fit(
        X_train_scaled, y_train,
        epochs=EPOCHS, batch_size=best_params['batch_size'],
        callbacks=[callbacks.EarlyStopping(monitor='loss', patience=PATIENCE,
                                           restore_best_weights=True, verbose=0)],
        verbose=VERBOSE
    )
    training_histories.append({'fold': fold_idx+1, 'history': history.history})

    y_train_pred = best_model.predict(X_train_scaled, verbose=0).flatten()
    y_test_pred  = best_model.predict(X_test_scaled,  verbose=0).flatten()

    train_class_results = classify_and_evaluate(y_train, y_train_pred, df_train)
    test_class_results  = classify_and_evaluate(y_test,  y_test_pred,  df_test)

    print(f"折叠 {fold_idx+1} | 最优参数: {best_params} | "
          f"epochs: {len(history.history['loss'])} | "
          f"训练MAE: {train_class_results['overall']['mae']:.5f} | "
          f"测试MAE: {test_class_results['overall']['mae']:.5f}")

    fold_result = {
        'fold': fold_idx+1, 'best_params': best_params,
        'train_metrics': train_class_results, 'test_metrics': test_class_results,
        'n_train': len(train_idx), 'n_test': len(test_idx),
        'n_epochs': len(history.history['loss']),
        'train_structures': outer_fold['train_structures'],
        'test_structures':  outer_fold['test_structures']
    }
    outer_results.append(fold_result)
    best_models.append({'fold': fold_idx+1, 'model': best_model,
                        'scaler': scaler, 'best_params': best_params})
    classification_results['fold_wise'][f'fold_{fold_idx+1}'] = {
        'train': train_class_results, 'test': test_class_results}

    for i in range(len(train_idx)):
        all_train_predictions.append({
            'fold': fold_idx+1, 'atom_index': int(train_idx[i]),
            'structure_id': df_train.iloc[i]['structure_id'],
            'D_E': float(df_train.iloc[i]['D_E']),
            'T_O': int(df_train.iloc[i]['T_O']), 'T_C': int(df_train.iloc[i]['T_C']),
            'T_Ti1': int(df_train.iloc[i]['T_Ti1']), 'T_Ti2': int(df_train.iloc[i]['T_Ti2']),
            'y_true': float(y_train[i]), 'y_pred': float(y_train_pred[i]),
            'error': float(y_train_pred[i]-y_train[i]),
            'abs_error': float(abs(y_train_pred[i]-y_train[i]))
        })
    for i in range(len(test_idx)):
        all_test_predictions.append({
            'fold': fold_idx+1, 'atom_index': int(test_idx[i]),
            'structure_id': df_test.iloc[i]['structure_id'],
            'D_E': float(df_test.iloc[i]['D_E']),
            'T_O': int(df_test.iloc[i]['T_O']), 'T_C': int(df_test.iloc[i]['T_C']),
            'T_Ti1': int(df_test.iloc[i]['T_Ti1']), 'T_Ti2': int(df_test.iloc[i]['T_Ti2']),
            'y_true': float(y_test[i]), 'y_pred': float(y_test_pred[i]),
            'error': float(y_test_pred[i]-y_test[i]),
            'abs_error': float(abs(y_test_pred[i]-y_test[i]))
        })

elapsed_time = time.time() - start_time
print(f"\n嵌套CV完成，耗时 {elapsed_time:.1f} 秒 ({elapsed_time/60:.1f} 分钟)")


折叠 1 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | epochs: 93 | 训练MAE: 0.01439 | 测试MAE: 0.01517
折叠 2 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | epochs: 100 | 训练MAE: 0.01827 | 测试MAE: 0.01863
折叠 3 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | epochs: 108 | 训练MAE: 0.01883 | 测试MAE: 0.01905
折叠 4 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64} | epochs: 96 | 训练MAE: 0.02967 | 测试MAE: 0.02949
折叠 5 | 最优参数: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 32} | epochs: 85 | 训练MAE: 0.02707 | 测试MAE: 0.02810

嵌套CV完成，耗时 9079.5 秒 (151.3 分钟)


In [3]:
# =============================================================================
# 计算平均性能，应用电中性校正，保存所有结果
# =============================================================================

def calculate_averaged_metrics(outer_results, dataset_type='test'):
    key      = f'{dataset_type}_metrics'
    averaged = {}
    for category in ['overall', 'edge', 'interior']:
        maes  = [r[key][category]['mae']  for r in outer_results]
        rmses = [r[key][category]['rmse'] for r in outer_results]
        r2s   = [r[key][category]['r2']   for r in outer_results]
        ns    = [r[key][category]['n_samples'] for r in outer_results]
        averaged[category] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }
    averaged['by_atom_type'] = {}
    for label in ATOM_TYPE_LABELS:
        maes  = [r[key]['by_atom_type'][label]['mae']  for r in outer_results]
        rmses = [r[key]['by_atom_type'][label]['rmse'] for r in outer_results]
        r2s   = [r[key]['by_atom_type'][label]['r2']   for r in outer_results]
        ns    = [r[key]['by_atom_type'][label]['n_samples'] for r in outer_results]
        averaged['by_atom_type'][label] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }
    return averaged

train_averaged = calculate_averaged_metrics(outer_results, 'train')
test_averaged  = calculate_averaged_metrics(outer_results, 'test')
classification_results['averaged'] = {'train': train_averaged, 'test': test_averaged}

print("\n5折CV平均性能（测试集，校正前）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

# 电中性校正
df_train_all = pd.DataFrame(all_train_predictions)
df_test_all  = pd.DataFrame(all_test_predictions)

df_train_all_corrected = apply_charge_neutrality_correction(df_train_all)
df_test_all_corrected  = apply_charge_neutrality_correction(df_test_all)

outer_results_corrected          = []
classification_results_corrected = {'fold_wise': {}, 'averaged': {}}

for fold_idx in range(len(outer_folds)):
    fold_num = fold_idx + 1
    df_train_fold_corr = df_train_all_corrected[df_train_all_corrected['fold'] == fold_num]
    df_test_fold_corr  = df_test_all_corrected[ df_test_all_corrected['fold']  == fold_num]

    train_class_corr = classify_and_evaluate(
        df_train_fold_corr['y_true'].values, df_train_fold_corr['y_pred_corrected'].values, df_train_fold_corr)
    test_class_corr  = classify_and_evaluate(
        df_test_fold_corr['y_true'].values,  df_test_fold_corr['y_pred_corrected'].values,  df_test_fold_corr)

    outer_results_corrected.append({'fold': fold_num, 'train_metrics': train_class_corr,
                                    'test_metrics': test_class_corr,
                                    'n_train': len(df_train_fold_corr), 'n_test': len(df_test_fold_corr)})
    classification_results_corrected['fold_wise'][f'fold_{fold_num}'] = {
        'train': train_class_corr, 'test': test_class_corr}

train_averaged_corrected = calculate_averaged_metrics(outer_results_corrected, 'train')
test_averaged_corrected  = calculate_averaged_metrics(outer_results_corrected, 'test')
classification_results_corrected['averaged'] = {'train': train_averaged_corrected,
                                                 'test':  test_averaged_corrected}

print("\n5折CV平均性能（测试集，校正后）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged_corrected[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

test_Q_before = df_test_all_corrected.groupby('structure_id')['Q_total_before'].first()
test_Q_after  = df_test_all_corrected.groupby('structure_id')['y_pred_corrected'].sum()
print(f"\n测试集校正前平均|Q_total|: {np.abs(test_Q_before).mean():.6f} e")
print(f"测试集校正后平均|Q_total|: {np.abs(test_Q_after).mean():.2e} e")

# 最终模型
best_fold_idx = np.argmin([r['test_metrics']['overall']['mae'] for r in outer_results])
final_params  = outer_results[best_fold_idx]['best_params']
print(f"\n最终模型参数（折叠{best_fold_idx+1}）: {final_params}")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
final_model    = create_dnn_model(X.shape[1], **{k: final_params[k] for k in
                                                  ['hidden_layers','dropout_rate','learning_rate']})
final_history  = final_model.fit(
    X_scaled_final, y, epochs=EPOCHS, batch_size=final_params['batch_size'],
    callbacks=[callbacks.EarlyStopping(monitor='loss', patience=PATIENCE,
                                       restore_best_weights=True, verbose=0)],
    verbose=VERBOSE
)
print(f"最终模型训练 epochs: {len(final_history.history['loss'])}")

# 保存预测CSV
for fold_num in range(1, 6):
    df_train_all[df_train_all['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_train_predictions.csv', index=False)
    df_test_all[df_test_all['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_test_predictions.csv', index=False)
    df_train_all_corrected[df_train_all_corrected['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_train_predictions_corrected.csv', index=False)
    df_test_all_corrected[df_test_all_corrected['fold'] == fold_num].to_csv(
        f'models/dnn/fold_{fold_num}_test_predictions_corrected.csv', index=False)

# 保存分类评分CSV
def build_classification_rows(classification_results, atom_type_labels):
    rows = []
    for fold_num in range(1, 6):
        fold_data = classification_results['fold_wise'][f'fold_{fold_num}']
        for dataset in ['train', 'test']:
            for category in ['overall', 'edge', 'interior']:
                d = fold_data[dataset][category]
                rows.append({'fold': fold_num, 'dataset': dataset,
                             'category': category, 'subcategory': 'all',
                             'mae': d['mae'], 'rmse': d['rmse'], 'r2': d['r2'], 'n_samples': d['n_samples']})
            for label in atom_type_labels:
                d = fold_data[dataset]['by_atom_type'][label]
                rows.append({'fold': fold_num, 'dataset': dataset,
                             'category': 'atom_type', 'subcategory': label,
                             'mae': d['mae'], 'rmse': d['rmse'], 'r2': d['r2'], 'n_samples': d['n_samples']})
    return rows

pd.DataFrame(build_classification_rows(classification_results, ATOM_TYPE_LABELS)).to_csv(
    'models/dnn/classification_scores.csv', index=False)
pd.DataFrame(build_classification_rows(classification_results_corrected, ATOM_TYPE_LABELS)).to_csv(
    'models/dnn/classification_scores_corrected.csv', index=False)

# 保存校正前后对比CSV
comparison_rows = []
for category_name, category_key in [('Overall','overall'), ('Edge','edge'), ('Interior','interior')]:
    mb = test_averaged[category_key]['mae_mean']
    ma = test_averaged_corrected[category_key]['mae_mean']
    comparison_rows.append({'category': category_name, 'subcategory': 'all',
        'mae_before': mb, 'mae_after': ma, 'mae_improvement_percent': (mb-ma)/mb*100,
        'mae_std_before': test_averaged[category_key]['mae_std'],
        'mae_std_after':  test_averaged_corrected[category_key]['mae_std']})
for label in ATOM_TYPE_LABELS:
    mb = test_averaged['by_atom_type'][label]['mae_mean']
    ma = test_averaged_corrected['by_atom_type'][label]['mae_mean']
    comparison_rows.append({'category': 'Atom_Type', 'subcategory': label,
        'mae_before': mb, 'mae_after': ma, 'mae_improvement_percent': (mb-ma)/mb*100,
        'mae_std_before': test_averaged['by_atom_type'][label]['mae_std'],
        'mae_std_after':  test_averaged_corrected['by_atom_type'][label]['mae_std']})
pd.DataFrame(comparison_rows).to_csv('models/dnn/correction_comparison.csv', index=False)

# 保存校正量统计CSV
edge_mask_all     = df_test_all_corrected['D_E'] < EDGE_THRESHOLD
interior_mask_all = ~edge_mask_all
correction_stats_rows = [{'category': 'overall', 'subcategory': 'all',
    'mean_correction':     float(df_test_all_corrected['correction_amount'].abs().mean()),
    'edge_correction':     float(df_test_all_corrected.loc[edge_mask_all,     'correction_amount'].abs().mean()),
    'interior_correction': float(df_test_all_corrected.loc[interior_mask_all, 'correction_amount'].abs().mean())}]
for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
    atom_mask          = df_test_all_corrected[col] == 1
    atom_edge_mask     = atom_mask & edge_mask_all
    atom_interior_mask = atom_mask & interior_mask_all
    correction_stats_rows.append({'category': 'atom_type', 'subcategory': label,
        'mean_correction':     float(df_test_all_corrected.loc[atom_mask,          'correction_amount'].abs().mean()),
        'edge_correction':     float(df_test_all_corrected.loc[atom_edge_mask,     'correction_amount'].abs().mean()) if atom_edge_mask.sum()     > 0 else float('nan'),
        'interior_correction': float(df_test_all_corrected.loc[atom_interior_mask, 'correction_amount'].abs().mean()) if atom_interior_mask.sum() > 0 else float('nan')})
pd.DataFrame(correction_stats_rows).to_csv('models/dnn/correction_stats.csv', index=False)

# 保存模型文件（DNN特有：.keras 格式）
final_model.save('models/dnn/dnn_model.keras')

model_package = {
    'scaler': scaler_final, 'feature_cols': feature_cols, 'target_col': target_col,
    'best_params': final_params, 'model_path': 'models/dnn/dnn_model.keras',
    'cv_performance': {'train': train_averaged, 'test': test_averaged},
    'cv_performance_corrected': {'train': train_averaged_corrected, 'test': test_averaged_corrected},
    'edge_threshold': EDGE_THRESHOLD, 'atom_type_labels': ATOM_TYPE_LABELS,
    'training_config': {'epochs': EPOCHS, 'patience': PATIENCE}
}
with open('models/dnn/dnn_model_package.pkl', 'wb') as f:
    pickle.dump(model_package, f)

all_folds_info = []
for model_info in best_models:
    fold_num = model_info['fold']
    model_info['model'].save(f'models/dnn/fold_{fold_num}_model.keras')
    all_folds_info.append({'fold': fold_num,
                           'model_path': f'models/dnn/fold_{fold_num}_model.keras',
                           'scaler': model_info['scaler'],
                           'best_params': model_info['best_params']})
with open('models/dnn/dnn_all_folds.pkl', 'wb') as f:
    pickle.dump(all_folds_info, f)

results_dict = {
    'model_name': 'Deep Neural Network',
    'n_folds': len(outer_folds), 'n_features': len(feature_cols),
    'training_time_seconds': float(elapsed_time),
    'edge_threshold': EDGE_THRESHOLD,
    'architecture': {'param_combinations': param_combinations,
                     'max_epochs': EPOCHS, 'early_stopping_patience': PATIENCE},
    'cv_results': {'train': train_averaged, 'test': test_averaged},
    'cv_results_corrected': {'train': train_averaged_corrected, 'test': test_averaged_corrected},
    'fold_details': outer_results,
    'training_histories': training_histories,
    'final_model': {'params': final_params, 'n_epochs': len(final_history.history['loss'])}
}
with open('models/dnn/dnn_results.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

mean_epochs = np.mean([r['n_epochs'] for r in outer_results])
print(f"\n平均训练 epochs: {mean_epochs:.1f}")
print("\n已保存: 预测CSV(校正前/后), classification_scores.csv, "
      "classification_scores_corrected.csv, correction_comparison.csv, "
      "correction_stats.csv, dnn_model.keras, dnn_model_package.pkl, "
      "fold_*_model.keras, dnn_all_folds.pkl, dnn_results.json")


5折CV平均性能（测试集，校正前）:
  overall    - MAE: 0.02209 ± 0.00566  RMSE: 0.03047 ± 0.00474  R²: 0.998541 ± 0.000448
  edge       - MAE: 0.03966 ± 0.00285  RMSE: 0.05844 ± 0.00771  R²: 0.992831 ± 0.001838
  interior   - MAE: 0.01998 ± 0.00636  RMSE: 0.02480 ± 0.00580  R²: 0.999034 ± 0.000431

5折CV平均性能（测试集，校正后）:
  overall    - MAE: 0.02209 ± 0.00567  RMSE: 0.02917 ± 0.00470  R²: 0.998661 ± 0.000437
  edge       - MAE: 0.03845 ± 0.00323  RMSE: 0.05614 ± 0.00769  R²: 0.993377 ± 0.001745
  interior   - MAE: 0.02013 ± 0.00632  RMSE: 0.02357 ± 0.00608  R²: 0.999118 ± 0.000448

测试集校正前平均|Q_total|: 1.528645 e
测试集校正后平均|Q_total|: 7.74e-16 e

最终模型参数（折叠1）: {'hidden_layers': [64, 32], 'dropout_rate': 0.2, 'learning_rate': 0.001, 'batch_size': 64}
最终模型训练 epochs: 105

平均训练 epochs: 96.4

已保存: 预测CSV(校正前/后), classification_scores.csv, classification_scores_corrected.csv, correction_comparison.csv, correction_stats.csv, dnn_model.keras, dnn_model_package.pkl, fold_*_model.keras, dnn_all_folds.pkl, dnn_results.json